# 02 — Firm-year analysis of `general_2016_2018.csv`

Follows on from `01-explore.ipynb`, which eyeballed the raw extract.

All logic lives in `src/`; this notebook only calls it:

| module | what it provides |
|---|---|
| `src.load` | read -> validate -> clean |
| `src.aggregate` | the (auditor firm, audit year) grain |
| `src.charts` | ranked bar panels per year |

## Load

`load_clean` runs the schema validation first, so nothing below executes
against an extract that fails `GeneralSchema`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from src.load import load_clean

df, dropped = load_clean(ROOT / "data" / "general_2016_2018.csv")
print(f"{len(df):,} rows x {df.shape[1]} cols; dropped {dropped}")
df.head()

## Top audit firms per year

Aggregation from `src/aggregate.py`, plotting from `src/charts.py` — the
notebook only calls them.

Note the frame above came from `load_clean`, which maps the `GSA_MIGRATION`
sentinel to NA. That is why `auditees_per_firm_year` counts distinct
`auditee_ein` rather than `auditee_uei`: the UEI column is the placeholder on
99.97% of rows, so a distinct count over it collapses to 1 for almost every
firm-year.

In [ ]:
from src.aggregate import auditees_per_firm_year, summarise_by_firm_year

per_year = auditees_per_firm_year(df)
print(f"{len(per_year):,} firm-years")
per_year.head(10)

In [ ]:
# the wide view: distinct-value counts for every column, per firm-year
summary = summarise_by_firm_year(df)
print(f"{summary.shape[0]:,} firm-years x {summary.shape[1]} columns")

cols = ["auditor_firm_name", "audit_year", "n_rows", "auditee_ein", "auditee_state", "entity_type"]
summary.nlargest(10, "n_rows")[cols]

## Chart

One panel per audit year, ranked largest to smallest. Panels share a y-scale
so bar heights compare across years; each panel is ranked independently, so
the firms shown may differ between panels.

In [ ]:
from src.charts import plot_firms_by_year

fig = plot_firms_by_year(
    per_year,
    value_column="n_auditees",
    top_n=15,
    output_path=ROOT / "reports" / "top_firms_by_year.png",
)
fig

## Scratch